In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/cleaned/fact_sales_cleaned.csv"
)

print("Python rows:", len(df))
print("Python unique transactions:", df["transaction_id"].nunique())

### Calculate Transactions

In [ ]:
transactions = df["transaction_id"].nunique()

transactions

### Calculate Units Sold

In [ ]:
units_sold = df["quantity"].sum()

units_sold

### Calculate Gross Sales

In [ ]:
gross_sales = df["gross_sales"].sum()

gross_sales

### Calculate Discounts

In [ ]:
discounts = df["discount_amount"].sum()

discounts

### Calculate Net Sales

In [ ]:
calculated_net_sales = (
    df["gross_sales"] - df["discount_amount"]
).sum()

calculated_net_sales

### Calculate Cost

In [ ]:
total_cost = df["cost_amount"].sum()

total_cost

### Calculate Gross Profit

In [ ]:
calculated_profit = (
    df["net_sales"] - df["cost_amount"]
).sum()

calculated_profit

### Calculate Gross Margin %

In [ ]:
gross_margin_pct = (
    calculated_profit
    / calculated_net_sales
    * 100
)

gross_margin_pct

### Calculate ATV (Average Transaction Value)

In [ ]:
atv = (
    calculated_net_sales
    / transactions
)

atv

### Calculate UPT (Units Per Transaction)

In [ ]:
upt = (
    units_sold
    / transactions
)

upt

### Identifiable active customers

In [ ]:
active_customers = (
    df.loc[
        df["customer_id"] != "UNKNOWN",
        "customer_id"
    ].nunique()
)

active_customers

### Calculate identifiable stores

In [ ]:
active_stores = (
    df.loc[
        df["store_id"] != "UNKNOWN",
        "store_id"
    ].nunique()
)

active_stores

### Calculate identifiable products

In [ ]:
active_products = (
    df.loc[
        df["product_id"] != "UNKNOWN",
        "product_id"
    ].nunique()
)

active_products

In [ ]:
stored_net_sales = df["net_sales"].sum()

print("Stored Net Sales:", stored_net_sales)
print("Calculated Net Sales:", calculated_net_sales)
print("Difference:", stored_net_sales - calculated_net_sales)

In [ ]:
stored_profit = df["profit_amount"].sum()

print("Stored Profit:", stored_profit)
print("Calculated Profit:", calculated_profit)
print("Difference:", stored_profit - calculated_profit)

In [ ]:
calculated_gross_sales = (
    df["quantity"] * df["unit_price"]
).sum()

print("Stored Gross Sales:", gross_sales)
print("Calculated Gross Sales:", calculated_gross_sales)
print(
    "Difference:",
    gross_sales - calculated_gross_sales
)

### Python KPI table

In [ ]:
python_kpis = pd.DataFrame({
    "KPI": [
        "Transactions",
        "Units Sold",
        "Gross Sales",
        "Discounts",
        "Net Sales",
        "Total Cost",
        "Gross Profit",
        "Gross Margin %",
        "Average Transaction Value",
        "Units Per Transaction",
        "Customers",
        "Stores",
        "Products"
    ],
    
    "Python": [
        transactions,
        units_sold,
        gross_sales,
        discounts,
        calculated_net_sales,
        total_cost,
        calculated_profit,
        gross_margin_pct,
        atv,
        upt,
        active_customers,
        active_stores,
        active_products
    ]
})

python_kpis

In [ ]:
import pyodbc

In [ ]:
import pyodbc

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=RetailDataQuality;"
    "UID=sa;"
    "PWD=abu638488@&;"
    "TrustServerCertificate=yes;"
)

### Read the SQL KPI view

In [ ]:
sql_kpis = pd.read_sql(
    "SELECT * FROM dbo.vw_kpi_reconciliation",
    connection
)

sql_kpis

### Build the recociliation table

In [ ]:
sql_values = sql_kpis.iloc[0]

reconciliation = pd.DataFrame({
    "KPI": python_kpis["KPI"],
    "Python": python_kpis["Python"],
})

In [ ]:
reconciliation["SQL Server"] = [
    sql_values["transactions"],
    sql_values["units_sold"],
    sql_values["gross_sales"],
    sql_values["discounts"],
    sql_values["net_sales"],
    sql_values["total_cost"],
    sql_values["gross_profit"],
    sql_values["gross_margin_pct"],
    sql_values["average_transaction_value"],
    sql_values["units_per_transaction"],
    sql_values["customers"],
    sql_values["stores"],
    sql_values["products"]
]

### Calculate differences

In [ ]:
reconciliation["Difference"] = (
    reconciliation["Python"]
    - reconciliation["SQL Server"]
)

In [ ]:
reconciliation["Status"] = reconciliation["Difference"].abs().apply(
    lambda x: "PASS" if x <= 0.01 else "FAIL"
)

In [ ]:
reconciliation